# Quick Ball Inference with DETR + OC-SORT + H.264 Preview

Run DETR ball detection on a SoccerNet Tracking sequence, pass cleaned detector candidates into an OC-SORT-style tracker, and render an H.264 preview.


In [5]:
# Install the H.264 ffmpeg helper if it is missing.
%pip install -q imageio-ffmpeg


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from pathlib import Path
import configparser
import csv
import json
import math
import os
import shutil
import subprocess
import sys
from typing import Any, Dict, List, Optional

import cv2
import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, Video, display
from tqdm.auto import tqdm


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    marker_sets = [
        ("data", "outputs", "notebooks"),
        ("data", "notebooks"),
        ("README.md", "notebooks"),
    ]
    for candidate in [current, *current.parents]:
        for markers in marker_sets:
            if all((candidate / marker).exists() for marker in markers):
                return candidate
    return current


def has_soccernet_sequences(path: Path) -> bool:
    path = Path(path)
    return path.exists() and any(path.rglob("seqinfo.ini"))


def resolve_kaggle_tracking_train_dir() -> Path:
    input_root = Path("/kaggle/input")
    candidates = [
        input_root / "notebooks" / "vtnhan2906" / "soccernet-tracking" / "train",
        input_root / "soccernet-tracking" / "train",
        input_root / "soccernet" / "tracking" / "train",
    ]
    for candidate in candidates:
        if has_soccernet_sequences(candidate):
            return candidate
    if input_root.exists():
        for seqinfo in input_root.rglob("seqinfo.ini"):
            sequence_dir = seqinfo.parent
            if sequence_dir.parent.name.lower() == "train":
                return sequence_dir.parent
    raise FileNotFoundError("Could not find SoccerNet tracking train split under /kaggle/input.")


def resolve_local_tracking_train_dir(project_root: Path) -> Path:
    candidates = [
        project_root / "data" / "raw" / "tracking" / "train",
        project_root / "data" / "tracking" / "train",
    ]
    for candidate in candidates:
        if has_soccernet_sequences(candidate):
            return candidate
    raise FileNotFoundError("Could not find local SoccerNet tracking train split under data/raw/tracking/train.")


def parse_seqinfo(seq_dir: Path) -> Dict[str, Any]:
    ini_path = seq_dir / "seqinfo.ini"
    if not ini_path.exists():
        raise FileNotFoundError(f"Missing seqinfo.ini in {seq_dir}")
    cp = configparser.ConfigParser()
    cp.read(ini_path)
    sec = cp["Sequence"]
    return {
        "name": sec.get("name", seq_dir.name),
        "im_dir": sec.get("imDir", "img1"),
        "frame_rate": sec.getint("frameRate", fallback=25),
        "seq_length": sec.getint("seqLength"),
        "im_width": sec.getint("imWidth"),
        "im_height": sec.getint("imHeight"),
        "im_ext": sec.get("imExt", ".jpg"),
    }


def frame_path(seq_dir: Path, frame_id: int) -> Path:
    info = parse_seqinfo(seq_dir)
    return seq_dir / info["im_dir"] / f"{frame_id:06d}{info['im_ext']}"


def discover_sequences(root: Path) -> List[Path]:
    seqs = []
    for seqinfo in Path(root).rglob("seqinfo.ini"):
        seq_dir = seqinfo.parent
        try:
            info = parse_seqinfo(seq_dir)
            if (seq_dir / info["im_dir"]).exists():
                seqs.append(seq_dir)
        except Exception:
            pass
    return sorted(set(seqs), key=lambda path: path.name)


def select_sequence(seqs: List[Path], sequence_name: Optional[str]) -> Path:
    if not seqs:
        raise FileNotFoundError(f"No sequences found under {DATA_ROOT}")
    if sequence_name:
        for seq in seqs:
            if seq.name == sequence_name:
                return seq
        raise FileNotFoundError(f"Sequence {sequence_name!r} not found under {DATA_ROOT}")
    return seqs[0]


def selected_frame_paths(seq_dir: Path, max_frames: Optional[int]) -> List[Path]:
    info = parse_seqinfo(seq_dir)
    paths = [frame_path(seq_dir, frame_id) for frame_id in range(1, info["seq_length"] + 1)]
    paths = [path for path in paths if path.exists()]
    return paths[:max_frames] if max_frames is not None else paths


def find_ffmpeg_executable():
    ffmpeg = shutil.which("ffmpeg")
    if ffmpeg:
        return ffmpeg
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return None


def transcode_to_h264(video_path: Path) -> Path:
    video_path = Path(video_path)
    ffmpeg = find_ffmpeg_executable()
    if ffmpeg is None:
        raise RuntimeError(
            "H.264 preview needs ffmpeg. Install one option, then rerun the render cell:\n"
            "  %pip install imageio-ffmpeg\n"
            "  or Windows: winget install Gyan.FFmpeg"
        )
    tmp_path = video_path.with_name(f"{video_path.stem}_h264_tmp{video_path.suffix}")
    cmd = [
        ffmpeg, "-y", "-i", str(video_path),
        "-c:v", "libx264", "-pix_fmt", "yuv420p",
        "-movflags", "+faststart", "-preset", "veryfast", "-crf", "23",
        "-an", str(tmp_path),
    ]
    completed = subprocess.run(cmd, text=True, capture_output=True)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"ffmpeg H.264 transcode failed for {video_path}")
    tmp_path.replace(video_path)
    return video_path


def make_video_writer(path: Path, fps: float, width: int, height: int):
    path.parent.mkdir(parents=True, exist_ok=True)
    width = width - 1 if width % 2 else width
    height = height - 1 if height % 2 else height
    writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    if not writer.isOpened():
        raise RuntimeError(f"Could not open video writer: {path}")
    return writer, (width, height)


def draw_ball_box(image, box_xyxy, score, color=(0, 255, 255)):
    x1, y1, x2, y2 = [int(round(v)) for v in box_xyxy]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(image.shape[1] - 1, x2), min(image.shape[0] - 1, y2)
    cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)
    cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
    cv2.circle(image, (cx, cy), 5, color, -1, cv2.LINE_AA)
    cv2.putText(image, f"ball {score:.2f}", (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    return image


def write_detection_csv(path: Path, rows: List[Dict[str, Any]]):
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = ["frame_id", "x", "y", "w", "h", "score", "class_id", "class_name"]
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


try:
    from scipy.optimize import linear_sum_assignment
except Exception:
    linear_sum_assignment = None


class OCSortTrackParams:
    def __init__(
        self,
        top_k=10,
        iou_threshold=0.02,
        max_center_distance=90.0,
        center_distance_weight=0.35,
        direction_weight=0.15,
        max_age=12,
        min_hits=1,
        max_tracks_per_frame=1,
    ):
        self.top_k = top_k
        self.iou_threshold = iou_threshold
        self.max_center_distance = max_center_distance
        self.center_distance_weight = center_distance_weight
        self.direction_weight = direction_weight
        self.max_age = max_age
        self.min_hits = min_hits
        self.max_tracks_per_frame = max_tracks_per_frame


def xywh_to_xyxy_array(boxes):
    boxes = np.asarray(boxes, dtype=np.float32)
    if boxes.size == 0:
        return boxes.reshape(0, 4)
    out = boxes.copy()
    out[:, 2] = out[:, 0] + out[:, 2]
    out[:, 3] = out[:, 1] + out[:, 3]
    return out


def iou_matrix_xywh(a_xywh, b_xywh):
    a = xywh_to_xyxy_array(a_xywh)
    b = xywh_to_xyxy_array(b_xywh)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    area_a = np.maximum(0.0, a[:, 2] - a[:, 0]) * np.maximum(0.0, a[:, 3] - a[:, 1])
    area_b = np.maximum(0.0, b[:, 2] - b[:, 0]) * np.maximum(0.0, b[:, 3] - b[:, 1])
    union = np.maximum(area_a[:, None] + area_b[None, :] - inter, 1e-9)
    return inter / union


def xywh_center(box):
    return np.asarray([box[0] + box[2] * 0.5, box[1] + box[3] * 0.5], dtype=np.float32)


class _OCSortLiteTrack:
    _next_id = 1

    def __init__(self, bbox_xywh, score, frame_id):
        self.id = _OCSortLiteTrack._next_id
        _OCSortLiteTrack._next_id += 1
        self.bbox = np.asarray(bbox_xywh, dtype=np.float32)
        self.score = float(score)
        self.last_observation = self.bbox.copy()
        self.previous_observation = None
        self.velocity = np.zeros(2, dtype=np.float32)
        self.time_since_update = 0
        self.hits = 1
        self.age = 1
        self.last_frame_id = int(frame_id)

    def predict(self):
        pred = self.bbox.copy()
        pred[0] += float(self.velocity[0])
        pred[1] += float(self.velocity[1])
        self.bbox = pred
        self.time_since_update += 1
        self.age += 1
        return self.bbox

    def update(self, bbox_xywh, score, frame_id):
        bbox_xywh = np.asarray(bbox_xywh, dtype=np.float32)
        old_center = xywh_center(self.last_observation)
        new_center = xywh_center(bbox_xywh)
        observed_velocity = new_center - old_center
        self.velocity = 0.7 * self.velocity + 0.3 * observed_velocity
        self.previous_observation = self.last_observation.copy()
        self.last_observation = bbox_xywh.copy()
        self.bbox = bbox_xywh.copy()
        self.score = float(score)
        self.time_since_update = 0
        self.hits += 1
        self.last_frame_id = int(frame_id)

    def direction(self):
        norm = float(np.linalg.norm(self.velocity))
        return self.velocity / norm if norm > 1e-6 else np.zeros(2, dtype=np.float32)


def _linear_assignment(cost):
    if cost.size == 0:
        return np.empty((0, 2), dtype=int)
    if linear_sum_assignment is not None:
        rows, cols = linear_sum_assignment(cost)
        return np.asarray(list(zip(rows, cols)), dtype=int)
    pairs = []
    used_rows, used_cols = set(), set()
    flat = [(float(cost[r, c]), r, c) for r in range(cost.shape[0]) for c in range(cost.shape[1])]
    for _, r, c in sorted(flat):
        if r not in used_rows and c not in used_cols:
            pairs.append((r, c))
            used_rows.add(r)
            used_cols.add(c)
    return np.asarray(pairs, dtype=int)


def _ocsort_associate(dets_xywh, tracks, trk_xywh, params):
    if len(dets_xywh) == 0 or len(tracks) == 0:
        return [], list(range(len(dets_xywh))), list(range(len(tracks)))

    ious = iou_matrix_xywh(dets_xywh, trk_xywh)
    det_centers = np.asarray([xywh_center(box) for box in dets_xywh], dtype=np.float32)
    trk_centers = np.asarray([xywh_center(box) for box in trk_xywh], dtype=np.float32)
    center_dist = np.linalg.norm(det_centers[:, None, :] - trk_centers[None, :, :], axis=2)
    center_cost = center_dist / max(float(params.max_center_distance), 1e-6)

    direction_cost = np.zeros_like(center_cost, dtype=np.float32)
    for t_idx, track in enumerate(tracks):
        direction = track.direction()
        if np.linalg.norm(direction) <= 1e-6:
            continue
        vectors = det_centers - trk_centers[t_idx]
        norms = np.linalg.norm(vectors, axis=1)
        valid = norms > 1e-6
        cos = np.zeros(len(dets_xywh), dtype=np.float32)
        cos[valid] = (vectors[valid] @ direction) / norms[valid]
        direction_cost[:, t_idx] = 1.0 - np.clip(cos, -1.0, 1.0)

    cost = (1.0 - ious) + params.center_distance_weight * center_cost + params.direction_weight * direction_cost
    assignments = _linear_assignment(cost)
    matches = []
    unmatched_dets = set(range(len(dets_xywh)))
    unmatched_trks = set(range(len(tracks)))
    for det_idx, trk_idx in assignments:
        valid_iou = float(ious[det_idx, trk_idx]) >= float(params.iou_threshold)
        valid_dist = float(center_dist[det_idx, trk_idx]) <= float(params.max_center_distance)
        if valid_iou or valid_dist:
            matches.append((int(det_idx), int(trk_idx)))
            unmatched_dets.discard(int(det_idx))
            unmatched_trks.discard(int(trk_idx))
    return matches, sorted(unmatched_dets), sorted(unmatched_trks)


def ocsort_track(cand_df: pd.DataFrame, frame_ids: List[int], params=None) -> pd.DataFrame:
    params = params or OCSortTrackParams()
    columns = ["frame_id", "track_id", "x", "y", "w", "h", "score", "class_id", "class_name", "detected"]
    if cand_df.empty:
        return pd.DataFrame(columns=columns)

    _OCSortLiteTrack._next_id = 1
    cand_df = cand_df.sort_values(["frame_id", "score"], ascending=[True, False]).reset_index(drop=True)
    tracks = []
    out_rows = []

    for frame_id in frame_ids:
        fc = cand_df[cand_df["frame_id"] == int(frame_id)].sort_values("score", ascending=False).head(int(params.top_k))
        dets = fc[["x", "y", "w", "h"]].to_numpy(dtype=np.float32) if len(fc) else np.zeros((0, 4), dtype=np.float32)
        det_scores = fc["score"].to_numpy(dtype=np.float32) if len(fc) else np.zeros((0,), dtype=np.float32)
        det_class_ids = fc["class_id"].to_numpy(dtype=np.int32) if len(fc) else np.zeros((0,), dtype=np.int32)
        det_class_names = fc["class_name"].astype(str).to_list() if len(fc) else []

        alive = []
        for track in tracks:
            track.predict()
            if track.time_since_update <= int(params.max_age):
                alive.append(track)
        tracks = alive

        trk_boxes = np.asarray([track.bbox for track in tracks], dtype=np.float32) if tracks else np.zeros((0, 4), dtype=np.float32)
        matches, unmatched_dets, _ = _ocsort_associate(dets, tracks, trk_boxes, params)

        for det_idx, trk_idx in matches:
            tracks[trk_idx].update(dets[det_idx], det_scores[det_idx], frame_id=frame_id)

        for det_idx in unmatched_dets:
            tracks.append(_OCSortLiteTrack(dets[det_idx], det_scores[det_idx], frame_id=frame_id))

        tracks = [track for track in tracks if track.time_since_update <= int(params.max_age)]
        frame_outputs = []
        for track in tracks:
            if track.time_since_update != 0 or track.hits < int(params.min_hits):
                continue
            x, y, w, h = [float(v) for v in track.bbox]
            class_id = 0
            class_name = "ball"
            if len(dets):
                centers = np.asarray([xywh_center(box) for box in dets], dtype=np.float32)
                idx = int(np.argmin(np.linalg.norm(centers - xywh_center(track.bbox), axis=1)))
                class_id = int(det_class_ids[idx])
                class_name = str(det_class_names[idx])
            frame_outputs.append({
                "frame_id": int(frame_id),
                "track_id": int(track.id),
                "x": x,
                "y": y,
                "w": w,
                "h": h,
                "score": float(track.score),
                "class_id": class_id,
                "class_name": class_name,
                "detected": 1,
            })

        frame_outputs = sorted(frame_outputs, key=lambda row: row["score"], reverse=True)
        if params.max_tracks_per_frame is not None:
            frame_outputs = frame_outputs[: int(params.max_tracks_per_frame)]
        out_rows.extend(frame_outputs)

    return pd.DataFrame(out_rows, columns=columns)


def write_track_csv(path: Path, track_df: pd.DataFrame):
    path.parent.mkdir(parents=True, exist_ok=True)
    columns = ["frame_id", "track_id", "x", "y", "w", "h", "score", "class_id", "class_name", "detected"]
    if track_df.empty:
        pd.DataFrame(columns=columns).to_csv(path, index=False)
    else:
        track_df[columns].to_csv(path, index=False)


def render_track_preview_h264(seq_dir: Path, frame_paths: List[Path], track_df: pd.DataFrame, video_path: Path, color=(0, 255, 255)):
    info = parse_seqinfo(seq_dir)
    first = cv2.imread(str(frame_paths[0]))
    if first is None:
        raise RuntimeError(f"Could not read first frame: {frame_paths[0]}")
    writer, size = make_video_writer(video_path, info["frame_rate"], first.shape[1], first.shape[0])
    by_frame = {int(fid): group for fid, group in track_df.groupby("frame_id")} if not track_df.empty else {}
    try:
        for frame_path_item in tqdm(frame_paths, desc=f"Render OC-SORT {seq_dir.name}"):
            frame_id = int(frame_path_item.stem)
            image = cv2.imread(str(frame_path_item))
            if image is None:
                continue
            frame_tracks = by_frame.get(frame_id)
            if frame_tracks is not None:
                for _, row in frame_tracks.iterrows():
                    x1, y1 = float(row.x), float(row.y)
                    x2, y2 = float(row.x + row.w), float(row.y + row.h)
                    draw_ball_box(image, (x1, y1, x2, y2), float(row.score), color=color)
                    cv2.putText(
                        image,
                        f"id {int(row.track_id)}",
                        (int(round(x1)), min(image.shape[0] - 8, int(round(y2)) + 16)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        color,
                        2,
                        cv2.LINE_AA,
                    )
            if image.shape[1] != size[0] or image.shape[0] != size[1]:
                image = cv2.resize(image, size, interpolation=cv2.INTER_AREA)
            writer.write(image)
    finally:
        writer.release()
    transcode_to_h264(video_path)
    return video_path


PROJECT_ROOT = find_project_root()
IS_KAGGLE = Path("/kaggle/working").exists()
DATA_ROOT = resolve_kaggle_tracking_train_dir() if IS_KAGGLE else resolve_local_tracking_train_dir(PROJECT_ROOT)
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
SEQUENCE_NAME = None  # Example: "SNMOT-060". None selects the first sequence.
MAX_FRAMES = 150      # Set None for full sequence.
CONF_THRESHOLD = 0.30  # DETR scores are low; 0.05 draws many false positives.
OCSORT_CANDIDATES_PER_FRAME = 5  # Candidate pool per frame before OC-SORT association.
OCSORT_MAX_TRACKS_PER_FRAME = 1  # Soccer has one ball; render the best active OC-SORT track per frame.
OCSORT_IOU_THRESHOLD = 0.02
OCSORT_MAX_CENTER_DISTANCE = 90.0
OCSORT_CENTER_DISTANCE_WEIGHT = 0.35
OCSORT_DIRECTION_WEIGHT = 0.15
OCSORT_MAX_AGE = 12
OCSORT_MIN_HITS = 1
NMS_IOU_THRESHOLD = 0.30
MIN_BOX_SIZE_PX = 2.0

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IS_KAGGLE:", IS_KAGGLE)
print("DATA_ROOT:", DATA_ROOT)
print("DEVICE:", DEVICE)

OUTPUT_ROOT = (Path("/kaggle/working") / "ball_detr_quick_infer") if IS_KAGGLE else PROJECT_ROOT / "outputs" / "ball_tracking" / "infer" / "detr_quick"
DETR_CHECKPOINT_CANDIDATES = [
    PROJECT_ROOT / "outputs" / "ball_tracking" / "detr_1x4_tracking_benchmark" / "detr_ball" / "checkpoint-best",
    PROJECT_ROOT / "outputs" / "ball_tracking" / "detr_1x4_tracking_benchmark" / "detr_ball" / "checkpoint-final",
]

def resolve_detr_checkpoint():
    for candidate in DETR_CHECKPOINT_CANDIDATES:
        if (candidate / "config.json").exists():
            return candidate
    raise FileNotFoundError("Missing DETR checkpoint. Expected outputs/ball_tracking/detr_1x4_tracking_benchmark/detr_ball/checkpoint-best")

DETR_CHECKPOINT = resolve_detr_checkpoint()
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("DETR_CHECKPOINT:", DETR_CHECKPOINT)


PROJECT_ROOT: C:\Users\CPU13374\Downloads\SoccerNet
IS_KAGGLE: False
DATA_ROOT: C:\Users\CPU13374\Downloads\SoccerNet\data\raw\tracking\train
DEVICE: cuda:0
OUTPUT_ROOT: C:\Users\CPU13374\Downloads\SoccerNet\outputs\ball_tracking\infer\detr_quick
DETR_CHECKPOINT: C:\Users\CPU13374\Downloads\SoccerNet\outputs\ball_tracking\detr_1x4_tracking_benchmark\detr_ball\checkpoint-best


In [7]:
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForObjectDetection


def load_detr_image_processor(path_or_name):
    try:
        return AutoImageProcessor.from_pretrained(str(path_or_name), use_fast=False)
    except TypeError:
        return AutoImageProcessor.from_pretrained(str(path_or_name))


def resolve_ball_label_ids(model) -> Optional[set[int]]:
    id2label = getattr(model.config, "id2label", {}) or {}
    ball_ids = {int(idx) for idx, label in id2label.items() if "ball" in str(label).lower()}
    return ball_ids or None


def box_iou_xyxy(box, boxes):
    if len(boxes) == 0:
        return np.asarray([], dtype=np.float32)
    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    box_area = max(0.0, float(box[2] - box[0])) * max(0.0, float(box[3] - box[1]))
    boxes_area = np.maximum(0.0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0.0, boxes[:, 3] - boxes[:, 1])
    union = np.maximum(box_area + boxes_area - inter, 1e-9)
    return inter / union


def nms_xyxy(boxes, scores, iou_threshold):
    if len(boxes) == 0:
        return []
    order = np.argsort(-scores)
    keep = []
    while len(order) > 0:
        current = int(order[0])
        keep.append(current)
        if len(order) == 1:
            break
        rest = order[1:]
        ious = box_iou_xyxy(boxes[current], boxes[rest])
        order = rest[ious <= iou_threshold]
    return keep


def clean_detr_detections(boxes, scores, labels, ball_ids):
    if len(boxes) == 0:
        return boxes, scores, labels

    keep_mask = np.ones(len(scores), dtype=bool)
    if ball_ids is not None:
        keep_mask &= np.asarray([int(label) in ball_ids for label in labels], dtype=bool)

    widths = boxes[:, 2] - boxes[:, 0]
    heights = boxes[:, 3] - boxes[:, 1]
    keep_mask &= (widths >= MIN_BOX_SIZE_PX) & (heights >= MIN_BOX_SIZE_PX)

    boxes = boxes[keep_mask]
    scores = scores[keep_mask]
    labels = labels[keep_mask]
    if len(boxes) == 0:
        return boxes, scores, labels

    keep = nms_xyxy(boxes, scores, NMS_IOU_THRESHOLD)
    boxes = boxes[keep]
    scores = scores[keep]
    labels = labels[keep]

    order = np.argsort(-scores)
    order = order[: int(OCSORT_CANDIDATES_PER_FRAME)]
    return boxes[order], scores[order], labels[order]


def run_detr_quick_inference():
    seqs = discover_sequences(DATA_ROOT)
    seq_dir = select_sequence(seqs, SEQUENCE_NAME)
    frame_paths = selected_frame_paths(seq_dir, MAX_FRAMES)
    if not frame_paths:
        raise FileNotFoundError(f"No frames found for {seq_dir}")

    scene_output = OUTPUT_ROOT / seq_dir.name
    cand_csv_path = scene_output / "detections.csv"
    track_csv_path = scene_output / "tracks_ocsort.csv"
    video_path = scene_output / "preview_ocsort.mp4"
    candidate_rows = []
    raw_detection_count = 0

    processor = load_detr_image_processor(DETR_CHECKPOINT)
    model = AutoModelForObjectDetection.from_pretrained(str(DETR_CHECKPOINT)).to(DEVICE)
    model.eval()
    ball_ids = resolve_ball_label_ids(model)
    id2label = {int(k): str(v) for k, v in (model.config.id2label or {}).items()}
    print("Sequence:", seq_dir.name, "frames:", len(frame_paths), "ball_label_ids:", ball_ids)

    with torch.no_grad():
        for frame_path_item in tqdm(frame_paths, desc=f"DETR detect {seq_dir.name}"):
            frame_id = int(frame_path_item.stem)
            pil_image = Image.open(frame_path_item).convert("RGB")
            inputs = processor(images=pil_image, return_tensors="pt").to(DEVICE)
            outputs = model(**inputs)
            target_sizes = torch.tensor([pil_image.size[::-1]], device=DEVICE)
            processed = processor.post_process_object_detection(
                outputs,
                threshold=CONF_THRESHOLD,
                target_sizes=target_sizes,
            )[0]

            boxes = processed["boxes"].detach().cpu().numpy()
            scores = processed["scores"].detach().cpu().numpy()
            labels = processed["labels"].detach().cpu().numpy().astype(int)
            raw_detection_count += len(scores)
            boxes, scores, labels = clean_detr_detections(boxes, scores, labels, ball_ids)
            for box, score, label in zip(boxes, scores, labels):
                x1, y1, x2, y2 = [float(v) for v in box.tolist()]
                if x2 <= x1 or y2 <= y1:
                    continue
                candidate_rows.append({
                    "frame_id": frame_id,
                    "x": x1,
                    "y": y1,
                    "w": x2 - x1,
                    "h": y2 - y1,
                    "score": float(score),
                    "class_id": int(label),
                    "class_name": id2label.get(int(label), "unknown"),
                })

    cand_df = pd.DataFrame(candidate_rows, columns=["frame_id", "x", "y", "w", "h", "score", "class_id", "class_name"])
    write_detection_csv(cand_csv_path, candidate_rows)

    frame_ids = [int(path.stem) for path in frame_paths]
    track_df = ocsort_track(cand_df, frame_ids=frame_ids, params=OCSortTrackParams(
        top_k=OCSORT_CANDIDATES_PER_FRAME,
        iou_threshold=OCSORT_IOU_THRESHOLD,
        max_center_distance=OCSORT_MAX_CENTER_DISTANCE,
        center_distance_weight=OCSORT_CENTER_DISTANCE_WEIGHT,
        direction_weight=OCSORT_DIRECTION_WEIGHT,
        max_age=OCSORT_MAX_AGE,
        min_hits=OCSORT_MIN_HITS,
        max_tracks_per_frame=OCSORT_MAX_TRACKS_PER_FRAME,
    ))
    write_track_csv(track_csv_path, track_df)
    render_track_preview_h264(seq_dir, frame_paths, track_df, video_path, color=(0, 220, 255))

    print("Raw DETR detections after score threshold:", raw_detection_count)
    print("Detector candidates for OC-SORT:", len(candidate_rows))
    print("OC-SORT track rows:", len(track_df))
    print("Detections:", cand_csv_path)
    print("Tracks:", track_csv_path)
    print("H.264 OC-SORT preview:", video_path)
    return {
        "sequence": seq_dir.name,
        "frames": len(frame_paths),
        "raw_detections": raw_detection_count,
        "candidates": len(candidate_rows),
        "tracks": len(track_df),
        "detections_csv": cand_csv_path,
        "tracks_csv": track_csv_path,
        "video": video_path,
    }

summary = run_detr_quick_inference()
summary


Loading weights: 100%|██████████| 530/530 [00:00<00:00, 8307.01it/s]


Sequence: SNMOT-060 frames: 150 ball_label_ids: {0}


Render OC-SORT SNMOT-060: 100%|██████████| 150/150 [00:02<00:00, 55.24it/s]


Raw DETR detections after score threshold: 1203
Detector candidates for OC-SORT: 402
OC-SORT track rows: 105
Detections: C:\Users\CPU13374\Downloads\SoccerNet\outputs\ball_tracking\infer\detr_quick\SNMOT-060\detections.csv
Tracks: C:\Users\CPU13374\Downloads\SoccerNet\outputs\ball_tracking\infer\detr_quick\SNMOT-060\tracks_ocsort.csv
H.264 OC-SORT preview: C:\Users\CPU13374\Downloads\SoccerNet\outputs\ball_tracking\infer\detr_quick\SNMOT-060\preview_ocsort.mp4


{'sequence': 'SNMOT-060',
 'frames': 150,
 'raw_detections': 1203,
 'candidates': 402,
 'tracks': 105,
 'detections_csv': WindowsPath('C:/Users/CPU13374/Downloads/SoccerNet/outputs/ball_tracking/infer/detr_quick/SNMOT-060/detections.csv'),
 'tracks_csv': WindowsPath('C:/Users/CPU13374/Downloads/SoccerNet/outputs/ball_tracking/infer/detr_quick/SNMOT-060/tracks_ocsort.csv'),
 'video': WindowsPath('C:/Users/CPU13374/Downloads/SoccerNet/outputs/ball_tracking/infer/detr_quick/SNMOT-060/preview_ocsort.mp4')}

In [8]:
display(HTML(f"<h3>DETR ball preview: {summary['sequence']}</h3>"))
display(Video(str(summary["video"]), embed=True, width=960))
